# FinShield Fraud Detection Platform — Modeling

This notebook develops and evaluates classification models for fraud detection on synthetic fintech transaction data.

## Modeling Objective
The objective is to predict whether a transaction is fraudulent (`is_fraud = 1`) using engineered behavioral, transactional, merchant, and customer-level features.

## Why this is a challenging problem
Fraud detection is a **binary classification problem with class imbalance**, where the positive class (fraud) is relatively rare. In this setting, standard accuracy is not sufficient, because a model could classify almost everything as legitimate and still appear to perform well.

## Modeling Strategy
This notebook follows a structured workflow:

1. Load the prepared modeling dataset
2. Split data into train and test sets using stratification
3. Build a preprocessing pipeline for mixed data types
4. Define a set of competing models
5. Compare models using cross-validation
6. Tune the best candidate(s)
7. Evaluate the selected model on the test set
8. Interpret business implications of the results

## Evaluation Metrics
The main metrics of interest are:

- **Precision (fraud class)**: how many flagged transactions are actually fraud
- **Recall (fraud class)**: how much fraud the model captures
- **F1-score (fraud class)**: balance between precision and recall
- **ROC AUC**: ranking quality across thresholds
- **PR AUC**: especially important in imbalanced classification problems

PR AUC is particularly relevant because fraud detection focuses on the minority class.

In [7]:
import sys
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    GridSearchCV,
    RandomizedSearchCV,
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    ConfusionMatrixDisplay,
)

warnings.filterwarnings("ignore")

project_root = Path.cwd().resolve().parent
sys.path.append(str(project_root))

from src.feature_engineering import get_modeling_data

## 1. Load Data and Prepare Modeling Inputs

We load the synthetic transaction dataset and transform it into a modeling-ready format using the project's feature engineering module.

This step ensures:
- engineered features are consistently created
- identifier columns are separated for traceability
- leakage variables are removed from the modeling matrix
- categorical and numerical feature groups are clearly defined

In [8]:
df = pd.read_csv("../data/raw/transactions.csv", parse_dates=["timestamp"])

X, y, entity_df, categorical_cols, numerical_cols = get_modeling_data(df)

print("X shape:", X.shape)
print("Fraud rate:", round(y.mean(), 4))
print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)

X shape: (100000, 39)
Fraud rate: 0.0333
Categorical columns: ['transaction_type', 'merchant_category', 'channel', 'device_type', 'transaction_city', 'customer_home_city', 'preferred_device_type']
Numerical columns: ['transaction_amount', 'merchant_risk_score', 'hour', 'day_of_week', 'is_weekend', 'customer_tenure_days', 'account_age_days', 'avg_amount_30d', 'std_amount_30d', 'transactions_last_7d', 'declines_last_30d', 'chargebacks_last_90d', 'amount_vs_avg_ratio', 'is_high_amount', 'is_very_high_amount', 'is_night_transaction', 'is_high_risk_merchant', 'is_new_device', 'is_foreign_transaction', 'channel_risk', 'category_risk', 'high_amount_new_device', 'foreign_high_risk_merchant', 'web_night_transaction', 'very_high_amount_foreign', 'log_transaction_amount', 'txn_velocity', 'customer_risk_score', 'decline_ratio', 'amount_merchant_risk', 'is_new_customer', 'abnormal_activity']


## 2. Train/Test Split

We split the data into training and testing sets using **stratification** on the target variable.

Why stratification matters:
- fraud is a minority class
- we want train and test to preserve approximately the same fraud rate
- this avoids misleading evaluation caused by target imbalance differences across splits

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train fraud rate:", round(y_train.mean(), 4))
print("Test fraud rate:", round(y_test.mean(), 4))

Train shape: (80000, 39)
Test shape: (20000, 39)
Train fraud rate: 0.0333
Test fraud rate: 0.0333


## 3. Preprocessing Pipeline

The dataset contains both numerical and categorical features, so we build a preprocessing pipeline using `ColumnTransformer`.

### Numerical features
- missing values: median imputation
- scaling: standardization

### Categorical features
- missing values: most frequent imputation
- encoding: one-hot encoding

Using a pipeline is critical because it:
- keeps preprocessing reproducible
- avoids leakage from manual transformations
- allows cross-validation and model tuning to operate correctly

In [4]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numerical_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

## 4. Define Competing Models

We evaluate several models with different levels of complexity:

### Models included
- **DummyClassifier**: trivial baseline for comparison
- **LogisticRegression**: interpretable linear baseline with class balancing
- **RandomForestClassifier**: nonlinear tree-based ensemble robust to interactions
- **HistGradientBoostingClassifier**: strong boosting model for tabular classification

### Why compare models?
A serious ML workflow should not assume the best model in advance. Instead, we compare multiple candidates under the same preprocessing and evaluation framework.

In [ ]:
models = {
    "dummy": DummyClassifier(strategy="prior"),
    "logistic_regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1,
    ),
    "hist_gradient_boosting": HistGradientBoostingClassifier(
        learning_rate=0.10,
        max_depth=6,
        max_iter=200,
        random_state=42,
    ),
}

models

In [5]:
logreg_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
    ]
)

logreg_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [6]:
y_pred_lr = logreg_pipeline.predict(X_test)
y_proba_lr = logreg_pipeline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_lr, digits=4, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_lr))
print("ROC AUC:", round(roc_auc_score(y_test, y_proba_lr), 4))
print("PR AUC:", round(average_precision_score(y_test, y_proba_lr), 4))

              precision    recall  f1-score   support

           0     0.9804    0.7314    0.8378     19334
           1     0.0687    0.5751    0.1227       666

    accuracy                         0.7262     20000
   macro avg     0.5245    0.6532    0.4803     20000
weighted avg     0.9500    0.7262    0.8140     20000

Confusion matrix:
 [[14141  5193]
 [  283   383]]
ROC AUC: 0.7007
PR AUC: 0.0836
